## 1. Environment Setup & Imports <a name='1-environment-setup'></a>

In [ ]:
# ─── Install required packages ────────────────────────────────────
import subprocess, sys

packages = [
    'pandas', 'numpy', 'matplotlib', 'seaborn', 'scikit-learn',
    'imbalanced-learn', 'xgboost', 'lightgbm', 'joblib', 'tqdm'
]

for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print(' All packages installed.')

In [ ]:
# ─── Imports ─────────────
import os
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from tqdm import tqdm

# ─── Sklearn ──────────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    f1_score, roc_auc_score, ConfusionMatrixDisplay
)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.inspection import permutation_importance
from sklearn.decomposition import PCA

# ─── Imbalanced-learn ────────────────────────────────────────────────────────
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline

# ─── Boosting libraries ───────────────────────────────────────────────────────
import xgboost as xgb
import lightgbm as lgb

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_style('whitegrid')
SEED = 42

print('All imports successful.')
print(f'   NumPy   : {np.__version__}')
print(f'   Pandas  : {pd.__version__}')
print(f'   XGBoost : {xgb.__version__}')
print(f'   LightGBM: {lgb.__version__}')

## 2. Data Loading & Merging <a name='2-data-loading'></a>

**How to get the data:**
1. Download from: https://unb.ca/cic/datasets/ids-2017.html
2. Download the **MachineLearningCSV.zip** (processed feature CSV files)
3. Extract and place all CSV files in a folder named `data/`
4. Update `DATA_DIR` below if needed

**OR** use Google Drive mount if running on Colab (see cell below).

In [ ]:


DATA_DIR = './data/MachineLearningCVE' 


EXPECTED_FILES = [
    'Monday-WorkingHours.pcap_ISCX.csv',
    'Tuesday-WorkingHours.pcap_ISCX.csv',
    'Wednesday-workingHours.pcap_ISCX.csv',
    'Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv',
    'Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv',
    'Friday-WorkingHours-Morning.pcap_ISCX.csv',
    'Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv',
    'Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv',
]

In [ ]:
# ─── Load & concatenate all CSV files ────────────────────────────────────────
def load_cicids(data_dir: str) -> pd.DataFrame:
    """Load all CSV files from data_dir into a single DataFrame."""
    csv_files = [f for f in os.listdir(data_dir) if f.endswith('.csv')]
    if not csv_files:
        raise FileNotFoundError(f'No CSV files found in: {data_dir}')
    
    frames = []
    for fname in tqdm(csv_files, desc='Loading CSVs'):
        path = os.path.join(data_dir, fname)
        df_tmp = pd.read_csv(path, encoding='utf-8', low_memory=False)
        df_tmp['source_file'] = fname   # track origin
        frames.append(df_tmp)
        print(f'  Loaded {fname:60s} → {df_tmp.shape[0]:>9,} rows')
    
    df = pd.concat(frames, axis=0, ignore_index=True)
    print(f'\n📊 Total shape after merge: {df.shape}')
    return df




DEMO_MODE = not os.path.isdir(DATA_DIR)

if DEMO_MODE:
    print('  DATA_DIR not found — running in DEMO mode with provided sample data.')
    print('    Replace the CSV string below with load_cicids(DATA_DIR) for the full run.\n')

    SAMPLE_CSV = """Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,Bwd Packet Length Mean,Bwd Packet Length Std,Flow Bytes/s,Flow Packets/s,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Total,Fwd IAT Mean,Fwd IAT Std,Fwd IAT Max,Fwd IAT Min,Bwd IAT Total,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Max,Bwd IAT Min,Fwd PSH Flags,Bwd PSH Flags,Fwd URG Flags,Bwd URG Flags,Fwd Header Length,Bwd Header Length,Fwd Packets/s,Bwd Packets/s,Min Packet Length,Max Packet Length,Packet Length Mean,Packet Length Std,Packet Length Variance,FIN Flag Count,SYN Flag Count,RST Flag Count,PSH Flag Count,ACK Flag Count,URG Flag Count,CWE Flag Count,ECE Flag Count,Down/Up Ratio,Average Packet Size,Avg Fwd Segment Size,Avg Bwd Segment Size,Fwd Header Length.1,Fwd Avg Bytes/Bulk,Fwd Avg Packets/Bulk,Fwd Avg Bulk Rate,Bwd Avg Bytes/Bulk,Bwd Avg Packets/Bulk,Bwd Avg Bulk Rate,Subflow Fwd Packets,Subflow Fwd Bytes,Subflow Bwd Packets,Subflow Bwd Bytes,Init_Win_bytes_forward,Init_Win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
49188,4,2,0,12,0,6,6,6,0,0,0,0,0,3000000,500000,4,0,4,4,4,4,0,4,4,0,0,0,0,0,0,0,0,0,40,0,500000,0,6,6,6,0,0,0,0,0,0,1,1,0,0,0,9,6,0,40,0,0,0,0,0,0,2,12,0,0,329,-1,1,20,0,0,0,0,0,0,0,0,BENIGN
80,40643,3,4,103,191,97,0,34.3333333333,54.3537793841,179,0,47.75,87.5457023503,7233.7179834166,172.2313805575,6773.8333333333,10209.4063963909,19958,3,20083,10041.5,14094.7594693915,20008,75,20685,6895,11413.0947599676,20069,3,0,0,0,0,72,92,73.8134488104,98.4179317472,0,179,36.75,66.282404474,4393.3571428572,0,0,0,1,0,0,0,0,1,42,34.3333333333,47.75,72,0,0,0,0,0,0,3,103,4,191,8192,913,2,20,0,0,0,0,0,0,0,0,BENIGN
389,18378,13,13,4160,5724,1672,0,320,612.9740342516,2634,0,440.3076923077,817.308936349,537816.955054957,1414.7350092502,735.12,3222.5996276299,16177,1,18378,1531.5,4628.9492131781,16177,3,18292,1524.3333333333,4674.3796347427,16337,1,0,0,0,0,284,284,707.3675046251,707.3675046251,0,2634,366.0740740741,700.4948210387,490692.994301994,0,0,0,1,0,0,0,0,1,380.1538461538,320,440.3076923077,284,0,0,0,0,0,0,13,4160,13,5724,8192,0,11,20,0,0,0,0,0,0,0,0,BENIGN
88,609,7,4,484,414,233,0,69.1428571429,111.9678950584,207,0,103.5,119.5115057223,1474548.44006568,18062.3973727422,60.9,115.1949555223,381,2,609,101.5,177.0895253819,460,2,467,155.6666666667,263.5608721592,460,3,0,0,0,0,164,104,11494.2528735632,6568.144499179,0,233,74.8333333333,107.5274454042,11562.1515151515,0,0,0,1,0,0,0,0,0,81.6363636364,69.1428571429,103.5,164,0,0,0,0,0,0,7,484,4,414,8192,2053,5,20,0,0,0,0,0,0,0,0,BENIGN
443,8439803,26,30,831,29479,249,0,31.9615384615,63.5115616367,2864,0,982.6333333333,945.5032771632,3591.3160532302,6.6352259644,153450.963636364,847487.633410855,6000678,1,6426835,257073.4,1196848.82984277,6000922,3,8389185,289282.24137931,1159789.05505012,6000678,1,0,0,0,0,540,612,3.0806406263,3.5545853381,0,2864,531.7543859649,833.476931738,694683.795739349,0,0,0,1,0,0,0,0,1,541.25,31.9615384615,982.6333333333,540,0,0,0,0,0,0,26,831,30,29479,29200,70,25,20,426053,0,426053,426053,6000678,0,6000678,6000678,BENIGN
"""

    import io
    df = pd.read_csv(io.StringIO(SAMPLE_CSV))
    print(f'Demo dataframe shape: {df.shape}')
else:
    df = load_cicids(DATA_DIR)

## 3. Exploratory Data Analysis (EDA) <a name='3-eda'></a>

In [ ]:
# ─── Basic information ────────────────────────────────────────────────────────
print('='*60)
print('DATASET OVERVIEW')
print('='*60)
print(f'Shape          : {df.shape}')
print(f'Memory usage   : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')
print(f'Columns        : {df.shape[1]}')
print()
df.info(verbose=False, show_counts=True)

In [ ]:
# ─── Display first rows ───────────────────────────────────────────────────────
df.head(3)

In [ ]:
# ─── Identify the label column ────────────────────────────────────────────────
# The CICIDS-2017 dataset uses ' Label' (with a leading space) or 'Label'
label_col = None
for c in df.columns:
    if c.strip().lower() == 'label':
        label_col = c
        break

print(f'Label column detected: "{label_col}"')

# Rename to clean name
df.rename(columns={label_col: 'Label'}, inplace=True)
label_col = 'Label'

# Strip column names of extra whitespace
df.columns = df.columns.str.strip()

# Clean label values
df['Label'] = df['Label'].str.strip()

In [ ]:
# ─── Class distribution ───────────────────────────────────────────────────────
label_counts = df['Label'].value_counts()
label_pct    = df['Label'].value_counts(normalize=True) * 100

label_summary = pd.DataFrame({
    'Count': label_counts,
    'Percentage (%)': label_pct.round(2)
})
print('CLASS DISTRIBUTION')
print(label_summary.to_string())

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Bar chart
colors = sns.color_palette('tab20', len(label_counts))
label_counts.plot(kind='bar', ax=axes[0], color=colors, edgecolor='black', linewidth=0.5)
axes[0].set_title('Sample Count per Class', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Attack Type')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=40, ha='right')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# Pie chart
label_pct.plot(kind='pie', ax=axes[1], autopct='%1.1f%%', startangle=140,
               colors=colors, pctdistance=0.85, labeldistance=1.05,
               textprops={'fontsize': 8})
axes[1].set_title('Class Proportions', fontsize=13, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(' Figure saved: class_distribution.png')

In [ ]:
# ─── Missing values ───────────────────────────────────────────────────────────
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
if missing.empty:
    print(' No missing values detected.')
else:
    print('Columns with missing values:')
    print(missing)
    
# ─── Infinite values ─────────────────────────────────────────────────────────
numeric_df = df.select_dtypes(include=[np.number])
inf_counts = np.isinf(numeric_df).sum()
inf_counts = inf_counts[inf_counts > 0]
if inf_counts.empty:
    print('No infinite values detected.')
else:
    print('\nColumns with Infinite values:')
    print(inf_counts)

In [ ]:
# ─── Statistical summary of numeric features ─────────────────────────────────
numeric_df.describe().T.sort_values('std', ascending=False).head(20)

In [ ]:
# ─── Distribution of top numeric features by class ───────────────────────────
# Pick a few informative features to visualise across attack types
PLOT_FEATURES = [
    'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets',
    'Flow Bytes/s', 'Flow Packets/s'
]

# Only keep features that exist in the dataframe
PLOT_FEATURES = [f for f in PLOT_FEATURES if f in df.columns]

if PLOT_FEATURES:
    fig, axes = plt.subplots(len(PLOT_FEATURES), 1, figsize=(14, 4 * len(PLOT_FEATURES)))
    if len(PLOT_FEATURES) == 1:
        axes = [axes]

    for ax, feat in zip(axes, PLOT_FEATURES):
        for label, grp in df.groupby('Label'):
            vals = grp[feat].replace([np.inf, -np.inf], np.nan).dropna()
            if len(vals) > 0:
                sns.kdeplot(vals.clip(vals.quantile(0.01), vals.quantile(0.99)),
                            ax=ax, label=label, fill=False, linewidth=1.5)
        ax.set_title(f'Distribution of "{feat}" by Attack Type', fontweight='bold')
        ax.set_xlabel(feat)
        ax.legend(fontsize=7, ncol=3)

    plt.tight_layout()
    plt.savefig('feature_distributions.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(' Figure saved: feature_distributions.png')

## 4. Data Preprocessing & Cleaning <a name='4-preprocessing'></a>

In [ ]:
# ─── 4.1 Drop non-feature columns ─────────────────────────────────────────────
COLUMNS_TO_DROP = ['source_file']   # added by loader; not a real feature
df.drop(columns=[c for c in COLUMNS_TO_DROP if c in df.columns], inplace=True)

# ─── 4.2 Replace Infinite values with NaN ────────────────────────────────────
df.replace([np.inf, -np.inf], np.nan, inplace=True)
print(f'Rows before dropping NaN: {len(df):,}')

# ─── 4.3 Drop rows with NaN ──────────────────────────────────────────────────
df.dropna(inplace=True)
print(f'Rows after  dropping NaN: {len(df):,}')

# ─── 4.4 Remove duplicate rows ───────────────────────────────────────────────
n_dupes = df.duplicated().sum()
print(f'Duplicate rows          : {n_dupes:,}')
df.drop_duplicates(inplace=True)
print(f'Rows after deduplication: {len(df):,}')

In [ ]:
# ─── 4.5 Unify label names (whitespace / case issues in the raw dataset) ──────
LABEL_ALIASES = {
    'Web Attack  Brute Force': 'Web Attack – Brute Force',
    'Web Attack  XSS': 'Web Attack – XSS',
    'Web Attack  Sql Injection': 'Web Attack – SQL Injection',
    'Web Attack – Sql Injection': 'Web Attack – SQL Injection',
}
df['Label'] = df['Label'].replace(LABEL_ALIASES)

print('Cleaned class distribution:')
print(df['Label'].value_counts().to_string())

In [ ]:
# ─── 4.6 Separate features and target ────────────────────────────────────────
X = df.drop(columns=['Label'])
y = df['Label']

# Keep only numeric columns (some versions have stray string cols)
X = X.select_dtypes(include=[np.number])

print(f'Feature matrix shape : {X.shape}')
print(f'Target shape         : {y.shape}')
print(f'Number of classes    : {y.nunique()}')
print(f'Classes              : {sorted(y.unique())}')

In [ ]:
# ─── 4.7 Encode labels ────────────────────────────────────────────────────────
le = LabelEncoder()
y_enc = le.fit_transform(y)

# Build a lookup table
class_map = dict(enumerate(le.classes_))
print('Label encoding:')
for code, name in sorted(class_map.items()):
    print(f'  {code:2d} → {name}')

## 5. Feature Engineering & Selection <a name='5-feature-engineering'></a>

In [ ]:
# ─── 5.1 Remove constant / near-zero-variance features ───────────────────────
from sklearn.feature_selection import VarianceThreshold

vt = VarianceThreshold(threshold=0.0)   # removes fully constant columns
X_vt = vt.fit_transform(X)
removed = X.columns[~vt.get_support()].tolist()
print(f'Removed {len(removed)} zero-variance features: {removed}')
X = pd.DataFrame(X_vt, columns=X.columns[vt.get_support()])
print(f'Features remaining: {X.shape[1]}')

In [ ]:
# ─── 5.2 Correlation analysis — remove highly correlated pairs ───────────────
CORR_THRESHOLD = 0.98

corr_matrix = X.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop_corr = [col for col in upper.columns if any(upper[col] > CORR_THRESHOLD)]

print(f'Features with correlation > {CORR_THRESHOLD}: {len(to_drop_corr)}')
print(to_drop_corr)
X.drop(columns=to_drop_corr, inplace=True)
print(f'Features remaining after corr filter: {X.shape[1]}')

In [ ]:
# ─── 5.3 Correlation heatmap ─────────────────────────────────────────────────
# Show heatmap of remaining features (sampled to 30 for legibility)
sample_feats = X.columns[:30].tolist()
fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(X[sample_feats].corr(), ax=ax, cmap='coolwarm', center=0,
            linewidths=0.3, annot=False)
ax.set_title('Feature Correlation Heatmap (first 30 features)', fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print(' Figure saved: correlation_heatmap.png')

In [ ]:
# ─── 5.4 Feature importance via Extra-Trees (fast, handles multiclass) ────────
print('Computing feature importances via ExtraTrees (this may take a minute)…')
et = ExtraTreesClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)
et.fit(X, y_enc)

feat_imp = pd.Series(et.feature_importances_, index=X.columns)\
             .sort_values(ascending=False)

TOP_K = 40
fig, ax = plt.subplots(figsize=(10, 10))
feat_imp.head(TOP_K).sort_values().plot(kind='barh', ax=ax, color='steelblue')
ax.set_title(f'Top-{TOP_K} Feature Importances (ExtraTrees)', fontweight='bold')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print(' Figure saved: feature_importance.png')

In [ ]:
# ─── 5.5 Keep only top-N most important features ─────────────────────────────
N_FEATURES = 50   # adjust as needed; more = better accuracy, slower training
TOP_FEATURES = feat_imp.head(N_FEATURES).index.tolist()

X_selected = X[TOP_FEATURES].copy()
print(f'Selected {N_FEATURES} features for modelling.')
print(TOP_FEATURES[:10], '...')

## 6. Handling Class Imbalance <a name='6-class-imbalance'></a>

In [ ]:
# ─── Train / Test split ───────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y_enc,
    test_size=0.20,
    random_state=SEED,
    stratify=y_enc
)
print(f'Train set : {X_train.shape[0]:,} samples')
print(f'Test  set : {X_test.shape[0]:,} samples')

In [ ]:
# ─── 6.1 Visualise class imbalance in train set ───────────────────────────────
train_label_counts = pd.Series(y_train).map(class_map).value_counts()

fig, ax = plt.subplots(figsize=(12, 4))
train_label_counts.plot(kind='bar', ax=ax, color=sns.color_palette('tab20', len(train_label_counts)))
ax.set_title('Class Distribution — Training Set (Before Resampling)', fontweight='bold')
ax.set_ylabel('Count')
ax.set_xlabel('Attack Type')
ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha='right')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig('class_imbalance_train.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── 6.2 Resampling strategy ──────────────────────────────────────────────────
# We use class_weight='balanced' inside tree-based models (no explicit resample)
# For models that don't support it, we use SMOTE on training data only.

# Compute class weights for use in model params
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)
weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, weights))
print('Sample class weights (first 5):', dict(list(class_weight_dict.items())[:5]))

# Scale features (needed for LR, SVM, KNN)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

## 7. Model Training <a name='7-model-training'></a>

We train **7 classifiers** and compare their performance:

| # | Model | Notes |
|---|-------|-------|
| 1 | Decision Tree | Fast baseline |
| 2 | Random Forest | Strong ensemble |
| 3 | Extra Trees | Faster RF variant |
| 4 | XGBoost | Gradient boosting |
| 5 | LightGBM | Fast gradient boosting |
| 6 | Logistic Regression | Linear baseline |
| 7 | K-Nearest Neighbours | Non-parametric |

In [ ]:
# ─── Model definitions ────────────────────────────────────────────────────────
N_CLASSES = len(le.classes_)

models = {
    'Decision Tree': DecisionTreeClassifier(
        max_depth=20,
        class_weight='balanced',
        random_state=SEED
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        class_weight='balanced',
        random_state=SEED,
        n_jobs=-1
    ),
    'Extra Trees': ExtraTreesClassifier(
        n_estimators=200,
        class_weight='balanced',
        random_state=SEED,
        n_jobs=-1
    ),
    'XGBoost': xgb.XGBClassifier(
        n_estimators=300,
        max_depth=8,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        use_label_encoder=False,
        eval_metric='mlogloss',
        random_state=SEED,
        n_jobs=-1
    ),
    'LightGBM': lgb.LGBMClassifier(
        n_estimators=300,
        max_depth=8,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        class_weight='balanced',
        random_state=SEED,
        n_jobs=-1,
        verbose=-1
    ),
    'Logistic Regression': LogisticRegression(
        max_iter=1000,
        class_weight='balanced',
        random_state=SEED,
        n_jobs=-1
    ),
    'KNN': KNeighborsClassifier(
        n_neighbors=5,
        n_jobs=-1
    ),
}

In [ ]:
# ─── Train all models ─────────────────────────────────────────────────────────
# Models that need scaled data
SCALED_MODELS = {'Logistic Regression', 'KNN'}

results = {}

for name, model in models.items():
    print(f'\n▶ Training: {name}…')
    
    X_tr = X_train_scaled if name in SCALED_MODELS else X_train
    X_te = X_test_scaled  if name in SCALED_MODELS else X_test
    
    model.fit(X_tr, y_train)
    y_pred = model.predict(X_te)
    
    acc  = accuracy_score(y_test, y_pred)
    f1_m = f1_score(y_test, y_pred, average='macro', zero_division=0)
    f1_w = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    
    results[name] = {
        'model'          : model,
        'y_pred'         : y_pred,
        'accuracy'       : acc,
        'f1_macro'       : f1_m,
        'f1_weighted'    : f1_w,
        'scaled'         : name in SCALED_MODELS,
    }
    print(f'   Accuracy: {acc:.4f}  |  F1-macro: {f1_m:.4f}  |  F1-weighted: {f1_w:.4f}')

print('\n All models trained.')

## 8. Model Evaluation & Comparison <a name='8-evaluation'></a>

In [ ]:
# ─── Summary table ────────────────────────────────────────────────────────────
summary = pd.DataFrame({
    name: {
        'Accuracy'     : v['accuracy'],
        'F1 Macro'     : v['f1_macro'],
        'F1 Weighted'  : v['f1_weighted'],
    }
    for name, v in results.items()
}).T.sort_values('F1 Macro', ascending=False)

print('MODEL COMPARISON SUMMARY')
print(summary.to_string())
summary

In [ ]:
# ─── Bar chart — model comparison ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(summary))
w = 0.25

ax.bar(x - w, summary['Accuracy'],    width=w, label='Accuracy',    color='steelblue')
ax.bar(x,     summary['F1 Macro'],    width=w, label='F1 Macro',    color='tomato')
ax.bar(x + w, summary['F1 Weighted'], width=w, label='F1 Weighted', color='seagreen')

ax.set_xticks(x)
ax.set_xticklabels(summary.index, rotation=30, ha='right')
ax.set_ylim(0, 1.05)
ax.set_title('Model Comparison — Accuracy & F1 Scores', fontsize=13, fontweight='bold')
ax.legend()
ax.set_ylabel('Score')

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(' Figure saved: model_comparison.png')

In [ ]:
# ─── Classification report for every model ───────────────────────────────────
for name, v in results.items():
    print('='*70)
    print(f'  {name}')
    print('='*70)
    print(classification_report(
        y_test, v['y_pred'],
        target_names=le.classes_,
        zero_division=0
    ))

## 9. Best Model — Deep Dive <a name='9-best-model'></a>

In [ ]:
# ─── Select best model by F1-macro ───────────────────────────────────────────
best_name = summary['F1 Macro'].idxmax()
best_result = results[best_name]
best_model  = best_result['model']

print(f'Best model: {best_name}')
print(f'   Accuracy   : {best_result["accuracy"]:.4f}')
print(f'   F1 Macro   : {best_result["f1_macro"]:.4f}')
print(f'   F1 Weighted: {best_result["f1_weighted"]:.4f}')

In [ ]:
# ─── Confusion Matrix ─────────────────────────────────────────────────────────
y_pred_best = best_result['y_pred']
cm = confusion_matrix(y_test, y_pred_best)

fig, ax = plt.subplots(figsize=(14, 11))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=le.classes_
)
disp.plot(ax=ax, cmap='Blues', colorbar=True, values_format='d',
          xticks_rotation='vertical')
ax.set_title(f'Confusion Matrix — {best_name}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print(' Figure saved: confusion_matrix.png')

In [ ]:
# ─── Normalised Confusion Matrix (recall per class) ──────────────────────────
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_, ax=ax)
ax.set_xlabel('Predicted', fontweight='bold')
ax.set_ylabel('True', fontweight='bold')
ax.set_title(f'Normalised Confusion Matrix — {best_name}', fontsize=13, fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.savefig('confusion_matrix_normalised.png', dpi=150, bbox_inches='tight')
plt.show()
print(' Figure saved: confusion_matrix_normalised.png')

In [ ]:
# ─── Per-class precision / recall / F1 bar chart ─────────────────────────────
from sklearn.metrics import precision_recall_fscore_support

prec, rec, f1, sup = precision_recall_fscore_support(
    y_test, y_pred_best, labels=range(N_CLASSES), zero_division=0
)

per_class_df = pd.DataFrame({
    'Precision': prec,
    'Recall'   : rec,
    'F1-Score' : f1,
    'Support'  : sup,
}, index=le.classes_)

print(per_class_df.to_string())

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(N_CLASSES)
w = 0.28

ax.bar(x - w, per_class_df['Precision'], w, label='Precision', color='steelblue')
ax.bar(x,     per_class_df['Recall'],    w, label='Recall',    color='tomato')
ax.bar(x + w, per_class_df['F1-Score'],  w, label='F1-Score',  color='seagreen')

ax.set_xticks(x)
ax.set_xticklabels(le.classes_, rotation=40, ha='right')
ax.set_ylim(0, 1.05)
ax.set_title(f'Per-Class Metrics — {best_name}', fontsize=13, fontweight='bold')
ax.set_ylabel('Score')
ax.legend()

plt.tight_layout()
plt.savefig('per_class_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: per_class_metrics.png')

## 10. Attack Type Visualization <a name='10-visualization'></a>

In [ ]:
# ─── 10.1 Actual vs Predicted attack distribution on test set ─────────────────
true_dist = pd.Series(y_test).map(class_map).value_counts()
pred_dist = pd.Series(y_pred_best).map(class_map).value_counts()

compare_df = pd.DataFrame({'Actual': true_dist, 'Predicted': pred_dist}).fillna(0)

fig, ax = plt.subplots(figsize=(14, 5))
compare_df.plot(kind='bar', ax=ax, color=['steelblue', 'tomato'])
ax.set_title('Actual vs Predicted Attack Type Counts', fontsize=13, fontweight='bold')
ax.set_xlabel('Attack Type')
ax.set_ylabel('Count')
ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha='right')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig('actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── 10.2 PCA 2D visualisation of attack clusters ────────────────────────────
print('Computing PCA 2D projection (may take a moment)…')

# Sample up to 20k points for speed
N_SAMPLE = min(20_000, len(X_test))
idx = np.random.default_rng(SEED).choice(len(X_test), N_SAMPLE, replace=False)

X_pca_input = X_test_scaled[idx] if best_result['scaled'] else X_test.values[idx]
y_pca = y_test[idx]

pca = PCA(n_components=2, random_state=SEED)
X_2d = pca.fit_transform(X_pca_input)

pca_df = pd.DataFrame(X_2d, columns=['PC1', 'PC2'])
pca_df['Label'] = pd.Series(y_pca).values
pca_df['Label_name'] = pca_df['Label'].map(class_map)

palette = sns.color_palette('tab20', N_CLASSES)
fig, ax = plt.subplots(figsize=(13, 9))
for code, name in class_map.items():
    mask = pca_df['Label'] == code
    ax.scatter(pca_df.loc[mask, 'PC1'], pca_df.loc[mask, 'PC2'],
               s=8, alpha=0.5, color=palette[code % 20], label=name)

ax.set_title('PCA 2D — Attack Type Clusters', fontsize=13, fontweight='bold')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)')
ax.legend(fontsize=7, ncol=2, markerscale=3)
plt.tight_layout()
plt.savefig('pca_clusters.png', dpi=150, bbox_inches='tight')
plt.show()
print(' Figure saved: pca_clusters.png')

In [ ]:
# ─── 10.3 Misclassification analysis ──────────────────────────────────────────
misclassified_mask = y_test != y_pred_best
n_misclassified = misclassified_mask.sum()
print(f'Total misclassified: {n_misclassified:,} / {len(y_test):,} ({100*n_misclassified/len(y_test):.2f}%)')

# Where are errors concentrated?
error_true  = pd.Series(y_test[misclassified_mask]).map(class_map).value_counts()
error_pred  = pd.Series(y_pred_best[misclassified_mask]).map(class_map).value_counts()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

error_true.plot(kind='bar', ax=axes[0], color='tomato')
axes[0].set_title('True Class of Misclassified Samples', fontweight='bold')
axes[0].set_xlabel('Attack Type')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=40, ha='right')

error_pred.plot(kind='bar', ax=axes[1], color='orange')
axes[1].set_title('Predicted Class of Misclassified Samples', fontweight='bold')
axes[1].set_xlabel('Attack Type')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=40, ha='right')

plt.tight_layout()
plt.savefig('misclassification_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── 10.4 Attack type timeline summary (requires source_file or timestamp) ────
# This shows per-file attack distribution if source_file exists

# Rebuild a summary dataframe from full dataset
attack_summary = df.groupby('Label').size().reset_index(name='count')
attack_summary['percentage'] = 100 * attack_summary['count'] / attack_summary['count'].sum()
attack_summary = attack_summary.sort_values('count', ascending=False)

print('\nFINAL ATTACK TYPE SUMMARY:')
print('='*55)
for _, row in attack_summary.iterrows():
    bar = '█' * int(row['percentage'] / 2)
    print(f'  {row["Label"]:40s} {row["count"]:>9,}  ({row["percentage"]:5.2f}%)  {bar}')
print('='*55)

## 11. Save & Export Best Model <a name='11-save-model'></a>

In [ ]:
# ─── Save model, scaler, encoder, feature list ───────────────────────────────
os.makedirs('saved_model', exist_ok=True)

artifacts = {
    'model'         : best_model,
    'scaler'        : scaler,
    'label_encoder' : le,
    'features'      : TOP_FEATURES,
    'best_model_name': best_name,
    'class_map'     : class_map,
}

joblib.dump(artifacts, 'saved_model/ids_pipeline.pkl', compress=3)
print(' Model artifacts saved to: saved_model/ids_pipeline.pkl')

# Save class summary CSV
per_class_df.to_csv('saved_model/per_class_metrics.csv')
summary.to_csv('saved_model/model_comparison.csv')
print('Metrics CSVs saved to saved_model/')

## 12. Inference on New Data <a name='12-inference'></a>

Use the cells below to load the saved pipeline and classify new network flows.

In [ ]:
# ─── Load saved artifacts ─────────────────────────────────────────────────────
loaded = joblib.load('saved_model/ids_pipeline.pkl')

model_inf   = loaded['model']
scaler_inf  = loaded['scaler']
le_inf      = loaded['label_encoder']
features_inf= loaded['features']
class_map_inf = loaded['class_map']

print(f' Loaded model: {loaded["best_model_name"]}')
print(f'   Expected features ({len(features_inf)}): {features_inf[:5]}…')

In [ ]:
# ─── Prediction function ──────────────────────────────────────────────────────
def predict_attack_type(raw_df: pd.DataFrame) -> pd.DataFrame:
    """
    Classify network flows.

    Parameters
    ----------
    raw_df : pd.DataFrame
        DataFrame with the same columns as the training data (Label column optional).

    Returns
    -------
    pd.DataFrame with columns: predicted_code, predicted_label, confidence
    """
    # Strip whitespace from column names
    raw_df = raw_df.copy()
    raw_df.columns = raw_df.columns.str.strip()

    # Keep only expected features
    missing_feats = [f for f in features_inf if f not in raw_df.columns]
    if missing_feats:
        raise ValueError(f'Missing features: {missing_feats}')

    X_new = raw_df[features_inf].copy()
    X_new = X_new.select_dtypes(include=[np.number])
    X_new.replace([np.inf, -np.inf], np.nan, inplace=True)
    X_new.fillna(0, inplace=True)

    # Scale if needed
    if loaded['best_model_name'] in SCALED_MODELS:
        X_new_arr = scaler_inf.transform(X_new)
    else:
        X_new_arr = X_new.values

    codes = model_inf.predict(X_new_arr)
    labels = le_inf.inverse_transform(codes)

    # Confidence (probability of top class)
    if hasattr(model_inf, 'predict_proba'):
        proba = model_inf.predict_proba(X_new_arr)
        conf = proba.max(axis=1)
    else:
        conf = np.ones(len(codes))

    return pd.DataFrame({
        'predicted_code' : codes,
        'predicted_label': labels,
        'confidence'     : conf,
    })


print(' predict_attack_type() function ready.')

In [ ]:
# ─── Demo prediction on a few test samples ───────────────────────────────────
# Rebuild X_test as a DataFrame for the function
X_test_df = pd.DataFrame(X_test.values if not isinstance(X_test, pd.DataFrame) else X_test,
                          columns=TOP_FEATURES)

sample = X_test_df.head(10).copy()
preds  = predict_attack_type(sample)

# Add true labels for comparison
preds['true_label'] = le.inverse_transform(y_test[:10])
preds['correct']    = preds['predicted_label'] == preds['true_label']

print('\nSample predictions (first 10 test flows):')
print(preds[['true_label', 'predicted_label', 'confidence', 'correct']].to_string())

In [ ]:
# ─── Final prediction distribution pie chart ──────────────────────────────────
all_preds = predict_attack_type(X_test_df)
pred_counts = all_preds['predicted_label'].value_counts()

fig, ax = plt.subplots(figsize=(10, 7))
wedges, texts, autotexts = ax.pie(
    pred_counts.values,
    labels=pred_counts.index,
    autopct='%1.1f%%',
    startangle=140,
    pctdistance=0.8,
    colors=sns.color_palette('tab20', len(pred_counts)),
    textprops={'fontsize': 8}
)
ax.set_title('Predicted Attack Type Distribution on Test Set', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('predicted_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(' Figure saved: predicted_distribution.png')

---

##  Summary

This notebook delivers a **complete Network Intrusion Detection System** pipeline:

| Step | What was done |
|------|---------------|
| Data Loading | Merged all 8 daily CSV files from CICIDS-2017 |
| EDA | Class distribution, missing/infinite values, feature distributions |
| Preprocessing | Cleaned labels, dropped NaN/Inf/duplicates |
| Feature Engineering | Variance filter, correlation filter, ExtraTrees importance |
| Class Imbalance | `class_weight='balanced'` in tree models; StandardScaler for linear models |
| Training | 7 classifiers: DT, RF, ExtraTrees, XGBoost, LightGBM, LR, KNN |
| Evaluation | Accuracy, F1-macro, F1-weighted, confusion matrix, per-class metrics |
| Visualization | PCA clusters, misclassification analysis, predicted distribution |
| Saving | Best model + scaler + encoder exported as `saved_model/ids_pipeline.pkl` |
| Inference | `predict_attack_type()` function ready for real-time use |

###  Next Steps
- **Hyperparameter tuning**: use `Optuna` or `GridSearchCV` on the best model
- **SHAP explanations**: `shap.TreeExplainer` to understand individual predictions
- **Real-time deployment**: wrap `predict_attack_type()` in a FastAPI endpoint
- **Feature engineering**: add packet rate ratios, temporal aggregations
- **Deep learning**: try a 1D CNN or LSTM on packet sequences